# Information & Management journal scraper

This notebook implements **Step 1** of a two-step scraper:

1. Scrape the *issues* listing page on ScienceDirect for the journal **Information & Management** from **2010 to present**.
2. (To be added) For each issue URL, scrape individual article metadata.

Step 1 collects the following fields for each issue:
- **Title** (issue description)
- **Volume** (volume number, e.g., `57`)
- **URL** (issue link on ScienceDirect)
- **Year** (issue year, e.g., `2019`)

Run the cells below from top to bottom after installing the required Python packages (`selenium`, `pandas`) and a compatible WebDriver (e.g., ChromeDriver).

In [ ]:
from selenium import webdriver
from selenium.webdriver.common.by import By
import time
import csv
import os
import re

BASE_URL = "https://www.sciencedirect.com"
ISSUES_URL = f"{BASE_URL}/journal/information-and-management/issues"
OUT_FILE = "information_and_management_issues_from_2010.csv"
MIN_YEAR = 2010

def extract_year(text: str):
    if not text:
        return None
    m = re.search(r"(19|20)\d{2}", text)
    return int(m.group(0)) if m else None

def extract_volume(text: str):
    if not text:
        return None
    m = re.search(r"[Vv]olume\s+(\d+)", text)
    return int(m.group(1)) if m else None

def write_header_if_needed(path):
    needs_header = not os.path.exists(path) or os.path.getsize(path) == 0
    if needs_header:
        with open(path, "w", newline="", encoding="utf-8") as f:
            writer = csv.writer(f)
            writer.writerow(["Title", "Volume", "URL", "Year"])

def append_rows(path, rows):
    if not rows:
        return
    with open(path, "a", newline="", encoding="utf-8") as f:
        writer = csv.writer(f)
        writer.writerows(rows)

# --- main ---
driver = webdriver.Chrome()
driver.get(ISSUES_URL)
driver.implicitly_wait(20)

# scroll to load more issues (tune iterations/sleep if needed)
last_height = driver.execute_script("return document.body.scrollHeight")
for _ in range(15):
    driver.execute_script("window.scrollTo(0, document.body.scrollHeight);")
    time.sleep(2)
    new_height = driver.execute_script("return document.body.scrollHeight")
    if new_height == last_height:
        break
    last_height = new_height

anchors = driver.find_elements(By.TAG_NAME, "a")
seen_urls = set()
rows = []

for a in anchors:
    href = a.get_attribute("href") or ""
    # issue links typically look like .../journal/information-and-management/vol/XX/issue/YY
    if "/journal/information-and-management/vol/" not in href:
        continue

    url = href if href.startswith("http") else BASE_URL.rstrip("/") + "/" + href.lstrip("/")
    if url in seen_urls:
        continue
    seen_urls.add(url)

    text = " ".join((a.text or "").split())
    # grab some surrounding text from parent for better year/volume detection
    try:
        parent = a.find_element(By.XPATH, "..")
        parent_text = " ".join((parent.text or "").split())
    except Exception:
        parent_text = text

    combined = f"{text} {parent_text}".strip()

    year = extract_year(combined)
    if year is None or year < MIN_YEAR:
        continue

    volume = extract_volume(combined)
    title = text or combined

    rows.append([title, volume, url, year])

driver.quit()

write_header_if_needed(OUT_FILE)
append_rows(OUT_FILE, rows)

print(f"Scraped {len(rows)} issues from {MIN_YEAR} onwards into {OUT_FILE}")

## STAGE 2

In [8]:
import pandas as pd
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
import csv
import time
import os

START_INDEX = 490
END_INDEX =  500

journals_data = pd.read_csv('Information_and_Management_Issues.csv')
OUT_FILE = 'Information_and_Management__article_data.csv'

def getAuthorsData(authors, driver):
    authdata = []
    for auth in authors:
        name, desc = '', ''
        email = None
        auth_data = None
        try:
            auth.click()
            time.sleep(2)
            auth_data = driver.find_element(By.CSS_SELECTOR, 'div.side-panel-content')
            auth_data_desc = auth_data.find_element(By.CSS_SELECTOR, 'div.affiliation')
            auth_name = auth_data.find_element(By.CSS_SELECTOR, 'div.author')
            name = auth_name.text.strip()
            desc = auth_data_desc.text.strip()
        except Exception as e:
            print("error at author data", e)
        try:
            if auth_data:
                auth_email = auth_data.find_element(By.CSS_SELECTOR, 'div.e-address a.anchor')
                email = auth_email.text.strip()
        except Exception:
            email = None
        authdata.append([name, email, desc])
    return authdata

if not os.path.exists(OUT_FILE) or os.path.getsize(OUT_FILE) == 0:
    with open(OUT_FILE, mode='a', newline='', encoding='utf-8') as file:
        writer = csv.writer(file)
        writer.writerow(['URL','Journal_Title','Article_Title','Volume_Issue','Month_Year','Abstract','Keywords','Author_name','Author_email','Author_Address'])

for index, row in journals_data.iloc[START_INDEX:END_INDEX].iterrows():
    driver = webdriver.Chrome()
    final_data = []

    url = str(row.get('URL', '')).strip()
    article_date = row.get('Vol Issue Year', None)

    if not url.startswith('http'):
        driver.quit()
        continue

    title = "N/A"
    article_journal = "N/A"
    article_vol = "N/A"
    abstract = None
    keyword_list = []

    try:
        driver.get(url)
        driver.implicitly_wait(10)

        WebDriverWait(driver, 10).until(
            EC.presence_of_element_located((By.CSS_SELECTOR, 'h1.Head'))
        )

        title = driver.find_element(By.CSS_SELECTOR, 'h1.Head').text.strip()
        article_details = driver.find_element(By.CSS_SELECTOR, 'div.Publication')
        article_journal = article_details.find_element(By.CSS_SELECTOR, 'h2.publication-title').text.strip()
        article_vol = article_details.find_element(By.CSS_SELECTOR, 'div.text-xs a.anchor').text.strip()
    except Exception as e:
        print(f"Error is {e} in this {url}")

    try:
        abstract = driver.find_element(By.ID, 'div.sp0050').text.strip()
    except Exception:
        abstract = None

    try:
        keywords = driver.find_elements(By.CSS_SELECTOR, 'div.keyword')
        keyword_list = []
        for key in keywords:
            keyword_list.append(key.text.strip())
    except Exception:
        keyword_list = []

    final_data = [url, article_journal, title, article_vol, article_date, abstract, keyword_list]

    try:
        author_group = driver.find_element(By.CSS_SELECTOR, 'div.author-group')
        authors = author_group.find_elements(By.CSS_SELECTOR, 'button.button-link')
        auth_data = getAuthorsData(authors, driver)
        if auth_data:
            for i in auth_data:
                with open(OUT_FILE, mode='a', newline='', encoding='utf-8') as file:
                    writer = csv.writer(file)
                    writer.writerow(final_data + i)
                    file.flush()
        else:
            with open(OUT_FILE, mode='a', newline='', encoding='utf-8') as file:
                writer = csv.writer(file)
                writer.writerow(final_data + ["N/A", "N/A", "N/A"])
                file.flush()
    except Exception as e:
        with open(OUT_FILE, mode='a', newline='', encoding='utf-8') as file:
            writer = csv.writer(file)
            writer.writerow(final_data + ["N/A", "N/A", "N/A"])
            file.flush()
        print(f"Error processing author data on {url}: {e}")

    driver.quit()
    time.sleep(3)
